In [9]:
import pandas as pd
import yfinance as yf
from prefect import task, flow
from ydata_profiling import ProfileReport
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

@task(name="descargar_datos_yahoo_finance")
def descargar_datos_yahoo_finance(tickers: list[str]) -> pd.DataFrame:
    """Descarga datos de Yahoo Finance para los tickers especificados."""
    print(f"Descargando datos para {tickers}...")
    df = yf.download(tickers, period='5y', progress=False)["Close"]
    print(f"✓ Datos descargados: {df.shape}")
    return df

@task(name="calcular_retornos")
def calcular_retornos(df: pd.DataFrame) -> pd.DataFrame:
    """Calcula los retornos porcentuales diarios."""
    print("Calculando retornos porcentuales...")
    retornos = df.pct_change().dropna()
    print(f"✓ Retornos calculados: {retornos.shape}")
    return retornos

@task(name="generar_reporte_profiling")
def generar_reporte_profiling(df: pd.DataFrame, titulo: str = "Reporte de Datos Financieros") -> ProfileReport:
    """Genera un reporte detallado usando ydata-profiling."""
    print(f"Generando reporte con ydata-profiling...")
    reporte = ProfileReport(df, title=titulo, explorative=True)
    print("✓ Reporte generado exitosamente")
    return reporte

@task(name="guardar_reporte_html")
def guardar_reporte_html(reporte: ProfileReport, archivo: str = "reporte_financiero.html") -> str:
    """Guarda el reporte como archivo HTML."""
    reporte.to_file(archivo)
    print(f"✓ Reporte guardado en: {archivo}")
    return archivo

@task(name="graficar_retornos")
def graficar_retornos(retornos: pd.DataFrame) -> dict:
    """Crea gráficas de retornos usando seaborn, incluyendo un pairplot."""
    print("Generando gráficas de retornos...")
    
    # Configurar estilo
    sns.set_style("whitegrid")
    plt.rcParams['figure.figsize'] = (14, 10)
    
    # 1. Pairplot de retornos
    print("  - Creando pairplot...")
    pairplot_fig = sns.pairplot(retornos, diag_kind='kde', plot_kws={'alpha': 0.6})
    pairplot_fig.savefig('pairplot_retornos.png', dpi=300, bbox_inches='tight')
    
    # 2. Heatmap de correlación
    print("  - Creando heatmap de correlación...")
    fig, ax = plt.subplots(figsize=(10, 8))
    correlacion = retornos.corr()
    sns.heatmap(correlacion, annot=True, fmt='.3f', cmap='coolwarm', center=0, 
                square=True, linewidths=1, cbar_kws={"shrink": 0.8}, ax=ax)
    plt.title('Matriz de Correlación de Retornos', fontsize=14, fontweight='bold')
    plt.tight_layout()
    fig.savefig('correlacion_retornos.png', dpi=300, bbox_inches='tight')
    
    # 3. Distribución de retornos por activo
    print("  - Creando distribuciones...")
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    
    for idx, ticker in enumerate(retornos.columns):
        sns.histplot(retornos[ticker], kde=True, ax=axes[idx], color='steelblue', bins=50)
        axes[idx].set_title(f'Distribución de Retornos - {ticker}', fontweight='bold')
        axes[idx].set_xlabel('Retorno Diario')
        axes[idx].set_ylabel('Frecuencia')
    
    plt.tight_layout()
    fig.savefig('distribucion_retornos.png', dpi=300, bbox_inches='tight')
    
    # 4. Series temporales de retornos
    print("  - Creando series temporales...")
    fig, ax = plt.subplots(figsize=(14, 6))
    for ticker in retornos.columns:
        ax.plot(retornos.index, retornos[ticker], label=ticker, alpha=0.7, linewidth=1.5)
    
    ax.set_title('Series Temporales de Retornos Diarios', fontsize=14, fontweight='bold')
    ax.set_xlabel('Fecha')
    ax.set_ylabel('Retorno Diario')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig('series_temporales_retornos.png', dpi=300, bbox_inches='tight')
    
    print("✓ Gráficas generadas exitosamente")
    
    return {
        'pairplot': 'pairplot_retornos.png',
        'correlacion': 'correlacion_retornos.png',
        'distribucion': 'distribucion_retornos.png',
        'series_temporales': 'series_temporales_retornos.png'
    }

@flow(name="descarga_datos_financieros")
def flujo_descarga_datos():
    """Flow principal que orquesta la descarga y análisis de datos."""
    tickers = ['F', 'TSLA', 'SPY', 'UEC']
    
    # Descargar datos
    datos = descargar_datos_yahoo_finance(tickers)
    
    # Calcular retornos
    retornos = calcular_retornos(datos)
    
    # Generar reporte
    reporte = generar_reporte_profiling(retornos)
    
    # Guardar reporte como HTML
    archivo = guardar_reporte_html(reporte)
    
    # Generar gráficas
    graficas = graficar_retornos(retornos)
    
    return datos, retornos, reporte, archivo, graficas

if __name__ == "__main__":
    resultado, retornos, reporte, archivo, graficas = flujo_descarga_datos()
    print("\n" + "="*50)
    print("Primeras filas del dataframe de precios:")
    print(resultado.head())
    print("\n" + "="*50)
    print("Primeras filas de retornos:")
    print(retornos.head())
    print("\n" + "="*50)
    print(f"Gráficas guardadas:")
    for grafica, archivo_grafica in graficas.items():
        print(f"  - {grafica}: {archivo_grafica}")

02:27:55.263 | INFO    | Flow run 'meteoric-ape' - Beginning flow run 'meteoric-ape' for flow 'descarga_datos_financieros'

02:27:55.265 | INFO    | Flow run 'meteoric-ape' - View at https://app.prefect.cloud/account/bcdd3fce-d2eb-4909-a6d6-d4c92de53389/workspace/2987211b-3429-4462-9b95-6aecb84a42a2/runs/flow-run/069d5bd2-affb-7b6a-8000-5ba30231af0e

Descargando datos para ['F', 'TSLA', 'SPY', 'UEC']...
✓ Datos descargados: (1255, 4)


02:27:55.629 | INFO    | Task run 'descargar_datos_yahoo_finance-3e7' - Finished in state Completed()

Calculando retornos porcentuales...
✓ Retornos calculados: (1254, 4)


...

02:27:59.756 | INFO    | Task run 'guardar_reporte_html-ab6' - Finished in state Completed()

Generando gráficas de retornos...
  - Creando pairplot...
  - Creando heatmap de correlación...
  - Creando distribuciones...
  - Creando series temporales...
✓ Gráficas generadas exitosamente


02:28:08.843 | INFO    | Task run 'graficar_retornos-dc2' - Finished in state Completed()

02:28:09.625 | INFO    | Flow run 'meteoric-ape' - Finished in state Completed()


Primeras filas del dataframe de precios:
Ticker             F         SPY        TSLA   UEC
Date                                              
2021-04-08  9.390280  381.550751  227.933334  3.31
2021-04-09  9.390280  384.324677  225.673340  3.09
2021-04-12  9.292699  384.464783  233.993332  2.85
2021-04-13  9.157586  385.604156  254.106674  2.81
2021-04-14  9.187610  384.287231  244.076660  2.88

Primeras filas de retornos:
Ticker             F       SPY      TSLA       UEC
Date                                              
2021-04-09  0.000000  0.007270 -0.009915 -0.066465
2021-04-12 -0.010392  0.000365  0.036867 -0.077670
2021-04-13 -0.014540  0.002964  0.085957 -0.014035
2021-04-14  0.003279 -0.003415 -0.039472  0.024911
2021-04-15  0.000000  0.010743  0.009041 -0.024306

Gráficas guardadas:
  - pairplot: pairplot_retornos.png
  - correlacion: correlacion_retornos.png
  - distribucion: distribucion_retornos.png
  - series_temporales: series_temporales_retornos.png


In [ ]:
# Mostrar las gráficas generadas
print("Visualizando gráficas generes...")
print("\n1. Pairplot de Retornos:")
from IPython.display import Image, display
display(Image('pairplot_retornos.png'))

print("\n2. Matriz de Correlación:")
display(Image('correlacion_retornos.png'))

print("\n3. Distribución de Retornos:")
display(Image('distribucion_retornos.png'))

print("\n4. Series Temporales de Retornos:")
display(Image('series_temporales_retornos.png'))